# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhardwaj-ayush03/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!pip -q install huggingface_hub duckdb
import duckdb
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

tok = userdata.get("HF_TOKEN").strip()          # .strip() removes stray spaces or newlines
print("token belongs to HF user:", HfApi().whoami(token=tok)["name"])

path = snapshot_download("FlyRank/internship-warehouse", repo_type="dataset", token=tok,
                         allow_patterns=["fact_content_daily_performance/month=2026-03/*"])

con = duckdb.connect()
con.execute(f"""CREATE TABLE m AS
  SELECT * FROM read_parquet('{path}/fact_content_daily_performance/month=2026-03/*.parquet')""")
print(con.sql("SELECT COUNT(*) AS rows_in_month FROM m").fetchone())

token belongs to HF user: ayushbh03


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378,)


## 1. Unit of analysis + time window

1. **What one row means:** in the warehouse table, one row = one page (content item) on one day for one client. For modeling, I roll it up to **one row per page** for the month.
2. **Table used:** `fact_content_daily_performance` (partition `month=2026-03`, a mid-panel month). The sealed final month (June 2026) is not touched.
3. **Time window:** features come from 2026-03-01 to 2026-03-14 (days 1-14). The outcome is measured on 2026-03-15 to 2026-03-31. The two windows do not overlap.
4. **Label / proxy (ranking task):** `declined_later = 1` if a page's average daily impressions in days 15-31 fall at least 30% below its average in days 1-14, among pages with at least 100 impressions in days 1-14 (minimum volume, to reduce noise). It is a proxy for decline, not proof of it. Success metric: Precision@50 on held-out clients.
5. **Deliberately excluded:** every metric from days 15-31 as a feature (that is the outcome window), and the hash ids as features (they are used only for grouping and the client-holdout split).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Decision moment:** the end of 2026-03-14. Every feature below is computed only from days 1-14, so all of it is knowable at that moment.

| Bucket | Field | Why |
|---|---|---|
| Feature | `early_impr` (total impressions, days 1-14) | knowable at the decision moment because it is summed only from days 1-14 |
| Feature | `early_ctr` (clicks / impressions, days 1-14) | knowable because both counts come from days 1-14 |
| Feature | `early_avg_pos` (mean `gsc_avg_position`, days 1-14) | knowable because it uses only days 1-14 |
| Feature | `early_active_days` (days with impressions, days 1-14) | knowable because it counts only days 1-14 |
| Feature | `early_momentum` (week 2 impressions vs week 1, both inside days 1-14) | knowable because both weeks end by day 14 |
| Label (proxy) | `declined_later` | 1 if average daily impressions in days 15-31 are at least 30% below days 1-14; a proxy, not proof of decline |
| Context | `client_hash_id`, `content_hash_id`, `gsc_data_available` | used for grouping, the client-holdout split, and availability filtering, not as features |
| Excluded | any column from days 15-31 (as a feature) | that is the outcome window; using it is leakage |
| Excluded | hash ids as features; `sessions_ai`, GA4 columns | ids carry no meaning; AI sessions are very sparse; GA4 is missing for many rows |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# Query 1: grain. Does one row really equal (report_date, client, content)?
con.sql("""
SELECT COUNT(*) AS total_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM m)) AS distinct_keys
FROM m""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys
0,9841378,9841378


In [6]:
# Query 2: slice size and date span
con.sql("""
SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages
FROM m""").df()

,rows,first_day,last_day,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


In [7]:
# Query 3: availability, filtered with IS TRUE
con.sql("""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
       ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ga4_available
FROM m""").df()

,total_rows,ga4_available_rows,pct_ga4_available
0,9841378,413966.0,4.2


In [10]:
import numpy as np, pandas as pd

pages = con.sql("""
WITH e AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS early_impr,
         SUM(gsc_clicks) AS early_clicks,
         AVG(gsc_avg_position) AS early_avg_pos,
         COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS early_active_days,
         SUM(CASE WHEN report_date >= DATE '2026-03-08' THEN gsc_impressions ELSE 0 END) AS wk2_impr,
         SUM(CASE WHEN report_date <  DATE '2026-03-08' THEN gsc_impressions ELSE 0 END) AS wk1_impr
  FROM m
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
  GROUP BY 1, 2
), l AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS late_impr
  FROM m
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-31'
  GROUP BY 1, 2
)
SELECT e.*, COALESCE(l.late_impr, 0) AS late_impr
FROM e LEFT JOIN l USING (client_hash_id, content_hash_id)
WHERE e.early_impr >= 100
""").df()

pages["early_ctr"] = pages["early_clicks"] / pages["early_impr"]
pages["early_momentum"] = (pages["wk2_impr"] + 1) / (pages["wk1_impr"] + 1)
pages["early_avg_pos"] = pages["early_avg_pos"].fillna(pages["early_avg_pos"].median())
pages["declined_later"] = ((pages["late_impr"] / 17) <= 0.7 * (pages["early_impr"] / 14)).astype(int)

feats = ["early_impr", "early_ctr", "early_avg_pos", "early_active_days", "early_momentum"]
print("pages:", len(pages), "| clients:", pages["client_hash_id"].nunique(),
      "| declined_later rate:", round(pages["declined_later"].mean(), 3))
pages[["client_hash_id", "content_hash_id"] + feats + ["declined_later"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 75526 | clients: 37 | declined_later rate: 0.254


,client_hash_id,content_hash_id,early_impr,early_ctr,early_avg_pos,early_active_days,early_momentum,declined_later
0,client_3197e6291363b4db,content_c9f4b2bf08fdba8f,109.0,0.000000,7.565858,14,1.413043,0
1,client_3197e6291363b4db,content_637c161a919d73aa,149.0,0.000000,27.077016,14,1.220588,0
2,client_3197e6291363b4db,content_847e51a122483aa3,251.0,0.003984,6.405946,14,1.162393,0
3,client_ff644d8251367cbb,content_ad81eee3c0435e0b,8769.0,0.006386,4.597238,14,1.037398,0
4,client_ff644d8251367cbb,content_18961035716d4ccf,3640.0,0.004670,4.621874,14,1.455833,0


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
              .split(pages, groups=pages["client_hash_id"]))

def quick_score(cols):
    m = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    m.fit(pages.iloc[tr][cols], pages.iloc[tr]["declined_later"])
    s = m.predict_proba(pages.iloc[te][cols])[:, 1]
    y = pages.iloc[te]["declined_later"].values
    order = np.argsort(-s)
    return {"Precision@50": round(y[order[:50]].mean(), 3), "ROC AUC": round(roc_auc_score(y, s), 3)}

print("HONEST (5 features):", quick_score(feats))

# THE TRAP: a column derived from the label's own outcome window
pages["leak_ratio"] = (pages["late_impr"] / 17) / (pages["early_impr"] / 14)
print("WITH LEAK          :", quick_score(feats + ["leak_ratio"]), " <- jumps toward perfect")

pages = pages.drop(columns="leak_ratio")   # delete it
print("LEAK REMOVED       :", quick_score(feats), " <- this is the honest number")

HONEST (5 features): {'Precision@50': np.float64(0.96), 'ROC AUC': np.float64(0.703)}
WITH LEAK          : {'Precision@50': np.float64(1.0), 'ROC AUC': np.float64(1.0)}  <- jumps toward perfect
LEAK REMOVED       : {'Precision@50': np.float64(0.96), 'ROC AUC': np.float64(0.703)}  <- this is the honest number


**What happened:** my feature frame has 75,526 pages from 37 clients (pages with at least 100 impressions in days 1-14 of March 2026). The proxy label `declined_later` is 1 for 25.4% of them. With the five early-window features (days 1-14 only), a depth-3 decision tree scored Precision@50 = 0.96 and ROC AUC = 0.703 on held-out clients (client-holdout split). Against the 0.254 base rate, the top 50 pages are far more often "declined later" than a random page, so the ranking has real signal at the top of the list, while the modest AUC shows that the model is far from separating all pages well.

**The trap:** I then deliberately added `leak_ratio`, a column built from the outcome window (days 15-31). Precision@50 rose to 1.00 and ROC AUC to 1.00, a perfect score. It looked perfect only because the column is the label in disguise: it tells the model the answer, so it teaches nothing. I deleted it, and the score returned to the honest 0.96 / 0.703, which is the number I keep.

**How I read it (directional, not proof):** the honest numbers describe one month, one client-holdout split and a 17-day outcome window, so small differences are noisy. The label is a proxy: some pages "decline" by chance (short-term swings, weekly patterns, regression to the mean), and I did not separate real decline from consolidation or seasonality. AUC is the steadier metric to track, and Precision@50 should always be read against the 0.254 base rate.

## 4. Data limits

The history is an unbalanced panel: clients started tracking at different times, so March 2026 contains only clients that already had data, and GA4 columns are missing for rows before a client's GA4 start (search-only rows). Results describe pages from the clients tracked in this month, not all clients, and a 17-day outcome window is short and can pick up noise or seasonality.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
con.sql("DESCRIBE m").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.